# 02.05 · Clustering & dimensionality reduction

**Chapter:** Machine Learning Fundamentals · **Milestone:** M1

## Goals
- [ ] k-means: how it works, choosing k (elbow, silhouette)
- [ ] PCA: explained variance, projection
- [ ] t-SNE vs UMAP for visualising high-dimensional data (you'll use these on embeddings in chapter 07)

## Resources
- [scikit-learn – Clustering](https://scikit-learn.org/stable/modules/clustering.html) · [Decomposition](https://scikit-learn.org/stable/modules/decomposition.html) — free
- [UMAP docs – How UMAP works](https://umap-learn.readthedocs.io/en/latest/how_umap_works.html) — free
- [Distill – How to use t-SNE effectively](https://distill.pub/2016/misread-tsne/) — free

In AI engineering you'll cluster user queries to find what people ask about, deduplicate datasets, and plot embeddings
to see whether your retriever separates topics.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_score

X, y = load_digits(return_X_y=True)  # 1797 images of 8x8 = 64 features
print(X.shape)
fig, axes = plt.subplots(1, 10, figsize=(10, 1.2))
for i, ax in enumerate(axes):
    ax.imshow(X[i].reshape(8, 8), cmap="gray_r"); ax.axis("off"); ax.set_title(y[i])
plt.show()

## 1. k-means
1. Pick k random centroids. 2. Assign each point to the nearest. 3. Move centroids to the mean of their points. Repeat.
Minimises within-cluster squared distance ("inertia"). You must choose k.

In [ ]:
ks = range(2, 16)
inertia, silhouette = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inertia.append(km.inertia_); silhouette.append(silhouette_score(X, km.labels_))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3))
a1.plot(ks, inertia, "o-"); a1.set_title("elbow: inertia vs k")
a2.plot(ks, silhouette, "o-"); a2.set_title("silhouette vs k (higher = better)")
plt.tight_layout(); plt.show()

km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(X)
print("agreement with true digits (ARI, 1 = perfect):", round(adjusted_rand_score(y, km.labels_), 3))
fig, axes = plt.subplots(1, 10, figsize=(10, 1.2))
for i, ax in enumerate(axes):
    ax.imshow(km.cluster_centers_[i].reshape(8, 8), cmap="gray_r"); ax.axis("off")
plt.suptitle("cluster centroids", y=1.1); plt.show()

## 2. PCA
Find orthogonal directions of maximum variance (eigenvectors of the covariance matrix, 01.03); keep the top few.
Linear, fast, deterministic. Good for compression and as a pre-step before clustering.

In [ ]:
pca = PCA().fit(X)
plt.plot(np.cumsum(pca.explained_variance_ratio_)); plt.axhline(0.9, ls="--", c="r")
plt.xlabel("components"); plt.ylabel("cumulative explained variance"); plt.show()
print("components for 90% variance:", np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.9) + 1, "of 64")

## 3. Visualising in 2-D: PCA vs t-SNE vs UMAP
- **PCA**: linear; preserves global structure; clusters often overlap.
- **t-SNE**: non-linear; great local clusters; distances *between* clusters and cluster sizes are meaningless; slow.
- **UMAP**: non-linear; usually faster than t-SNE and keeps more global structure; has a `transform` for new points.

In [ ]:
import umap

projections = {}
for name, reducer in {
    "PCA": PCA(n_components=2),
    "t-SNE": TSNE(n_components=2, random_state=0),
    "UMAP": umap.UMAP(n_components=2, random_state=0),
}.items():
    start = time.perf_counter()
    projections[name] = (reducer.fit_transform(X), time.perf_counter() - start)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, (Z, secs)) in zip(axes, projections.items()):
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=4)
    ax.set_title(f"{name} ({secs:.1f}s)"); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=axes, ticks=range(10)); plt.show()

## Exercises
1. Cluster after reducing to 20 PCA components. Does ARI improve? Is it faster?
2. Try `sklearn.cluster.HDBSCAN` (no k needed, finds noise points). Compare with k-means on the UMAP projection.
3. Write 30 short sentences on 3 topics (sports, cooking, programming). Represent each with `TfidfVectorizer`, cluster with k=3,
   and check if clusters match topics. In chapter 07 you'll redo this with real embeddings and see the difference.

In [ ]:
# your code here